# 📡 Project 13.2 — Huffman Signal Compressor
**DSA for Mechatronics · Week 13 — Greedy Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A telemetry system transmits sensor event codes over a low-bandwidth radio link.
Each code appears with a different frequency. We build a **Huffman code** to
assign shorter bit patterns to more frequent codes, minimising total bits sent.

1. **Build Huffman tree** — min-heap greedy: always merge the two lowest-frequency
   nodes, until one root remains
2. **Assign codes** — traverse tree: left edge = 0, right edge = 1
3. **Encode and decode** a sample message and verify round-trip
4. **Compare** against fixed-length encoding — measure compression ratio


## Step 1 — Generate signal frequency table

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_SYMBOLS  = random.randint(6, 10)
MSG_LEN    = random.randint(60, 100)

SYMBOLS    = random.sample(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"), N_SYMBOLS)
raw_freq   = [random.randint(5, 50) for _ in range(N_SYMBOLS)]
FREQ       = dict(zip(SYMBOLS, raw_freq))

# Generate a sample message consistent with frequencies
total_f    = sum(raw_freq)
MESSAGE    = "".join(random.choices(SYMBOLS, weights=raw_freq, k=MSG_LEN))

print(f"Signal compressor parameters:")
print(f"  Symbols   : {N_SYMBOLS}")
print(f"  Message   : {MSG_LEN} chars")
print()
print(f"  Symbol frequency table:")
print(f"  {'Symbol':>8}  {'Frequency':>10}  {'Proportion':>12}")
print(f"  {'─'*8}  {'─'*10}  {'─'*12}")
for sym in sorted(FREQ, key=lambda x: -FREQ[x]):
    prop = FREQ[sym] / total_f
    print(f"  {sym:>8}  {FREQ[sym]:>10}  {prop:>11.1%}")
print()
print(f"  Sample message (first 40 chars): {MESSAGE[:40]}...")


## Step 2 — Build Huffman tree with min-heap

In [ ]:
class HuffNode:
    """Node in the Huffman tree."""
    def __init__(self, sym, freq, left=None, right=None):
        self.sym   = sym
        self.freq  = freq
        self.left  = left
        self.right = right
    def __lt__(self, other):       # needed for heapq comparison
        return self.freq < other.freq

def build_huffman_tree(freq_table):
    """
    Build Huffman tree using a min-heap.

    Algorithm:
      1. Create one leaf node per symbol, push all to min-heap.
      2. Repeat until one node remains:
         a. Pop two nodes with lowest frequency (greedy minimum choice).
         b. Create internal node with freq = sum of both.
         c. Push new node back.
      3. The last remaining node is the tree root.

    Why greedy works: merging the two least frequent nodes last means
    their symbols get the LONGEST codes — exactly what we want for rare
    symbols. Exchange argument: swapping any pair of nodes cannot reduce
    total bit length (Huffman proved this is globally optimal).

    O(n log n) — n heap pops/pushes of O(log n) each.
    """
    heap = [HuffNode(sym, freq) for sym, freq in freq_table.items()]
    heapq.heapify(heap)
    merge_log = []
    step = 0
    while len(heap) > 1:
        step += 1
        left  = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = HuffNode(None, left.freq + right.freq, left, right)
        heapq.heappush(heap, merged)
        l_label = left.sym  if left.sym  else f"({left.freq})"
        r_label = right.sym if right.sym else f"({right.freq})"
        merge_log.append(f"  step {step}: merge {l_label}({left.freq}) + "
                         f"{r_label}({right.freq}) → ({merged.freq})")
    return heap[0], merge_log

def assign_codes(root):
    """Traverse tree; left=0, right=1. Returns {symbol: code_string}."""
    codes = {}
    def dfs(node, prefix):
        if node is None: return
        if node.sym is not None:
            codes[node.sym] = prefix if prefix else "0"  # single symbol edge case
            return
        dfs(node.left,  prefix + "0")
        dfs(node.right, prefix + "1")
    dfs(root, "")
    return codes

root, merge_log = build_huffman_tree(FREQ)
CODES = assign_codes(root)

print(f"Huffman tree construction:")
print(f"  Merge sequence:")
for line in merge_log:
    print(line)
print()
print(f"  Code table:")
print(f"  {'Symbol':>8}  {'Freq':>6}  {'Code':>14}  {'Bits':>5}")
print(f"  {'─'*8}  {'─'*6}  {'─'*14}  {'─'*5}")
for sym in sorted(CODES, key=lambda x: (len(CODES[x]), x)):
    print(f"  {sym:>8}  {FREQ[sym]:>6}  {CODES[sym]:>14}  {len(CODES[sym]):>5}")
import math
fixed_bits = math.ceil(math.log2(N_SYMBOLS)) if N_SYMBOLS > 1 else 1
print()
print(f"  Fixed-length bits needed : {fixed_bits} per symbol")
avg_huff = sum(FREQ[s] * len(CODES[s]) for s in CODES) / sum(FREQ.values())
print(f"  Huffman avg bits/symbol  : {avg_huff:.3f}")


## Step 3 — Encode and decode message, verify round-trip

In [ ]:
def encode(message, codes):
    """Encode message string to bitstring using code table."""
    return "".join(codes[c] for c in message)

def decode(bitstring, root):
    """
    Decode a Huffman-encoded bitstring back to the original message.
    Traverse tree from root: 0 → go left, 1 → go right.
    When a leaf is reached, emit its symbol and return to root.
    O(n) where n = length of bitstring.
    """
    result = []
    node   = root
    for bit in bitstring:
        node = node.left if bit == "0" else node.right
        if node.sym is not None:
            result.append(node.sym)
            node = root
    return "".join(result)

encoded   = encode(MESSAGE, CODES)
decoded   = decode(encoded, root)

huff_bits  = len(encoded)
fixed_msg_bits = len(MESSAGE) * fixed_bits
ratio = fixed_msg_bits / huff_bits if huff_bits > 0 else 1.0

print(f"Encode / Decode round-trip:")
print(f"  Message length        : {len(MESSAGE)} chars")
print(f"  Encoded bits          : {huff_bits}")
print(f"  Fixed-length bits     : {fixed_msg_bits} ({fixed_bits} bits × {len(MESSAGE)})")
print(f"  Compression ratio     : {ratio:.3f}× (Huffman is {(1-1/ratio)*100:.1f}% smaller)")
print()
print(f"  Encoded (first 60 bits): {encoded[:60]}...")
print(f"  Decoded (first 40 chars): {decoded[:40]}...")
print()
match = decoded == MESSAGE
print(f"  Round-trip correct    : {'✅ YES' if match else '❌ NO'}")


## Step 4 — Prefix-free verification + entropy lower bound

In [ ]:
def is_prefix_free(codes):
    """
    Check that no code is a prefix of another code.
    A valid Huffman code is always prefix-free (uniquely decodable).
    """
    code_list = list(codes.values())
    for i, c1 in enumerate(code_list):
        for j, c2 in enumerate(code_list):
            if i != j and c2.startswith(c1):
                return False, (c1, c2)
    return True, None

prefix_free, violation = is_prefix_free(CODES)

# Shannon entropy (theoretical minimum bits per symbol)
import math
total_f = sum(FREQ.values())
entropy = -sum((f/total_f) * math.log2(f/total_f) for f in FREQ.values() if f > 0)

# Kraft inequality: sum(2^{-len(code)}) ≤ 1 for prefix-free codes
kraft = sum(2**(-len(c)) for c in CODES.values())

print(f"Prefix-free and information-theoretic analysis:")
print()
print(f"  Prefix-free check     : {'✅ YES — no code is prefix of another' if prefix_free else f'❌ VIOLATION: {violation}'}")
print(f"  Kraft inequality      : Σ 2^{{-|code|}} = {kraft:.6f}  ({'✅ ≤ 1' if kraft <= 1.0001 else '❌ > 1'})")
print()
print(f"  Shannon entropy       : {entropy:.4f} bits/symbol (theoretical minimum)")
print(f"  Huffman avg bits/sym  : {avg_huff:.4f} bits/symbol")
print(f"  Gap from entropy      : {avg_huff - entropy:.4f} bits/symbol")
print(f"  (Huffman is optimal — always within 1 bit of entropy)")
print()
print(f"  Per-symbol summary:")
print(f"  {'Symbol':>6}  {'Freq':>6}  {'Prob':>7}  {'Code':>14}  {'Huff':>5}  {'Fixed':>6}")
print(f"  {'─'*6}  {'─'*6}  {'─'*7}  {'─'*14}  {'─'*5}  {'─'*6}")
for sym in sorted(CODES, key=lambda x: -FREQ[x]):
    p = FREQ[sym]/total_f
    print(f"  {sym:>6}  {FREQ[sym]:>6}  {p:>7.3f}  {CODES[sym]:>14}  "
          f"{len(CODES[sym]):>5}  {fixed_bits:>6}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " HUFFMAN SIGNAL COMPRESSOR — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  HUFFMAN TREE" + " "*(W-14) + "║")
print(f"║  {'Symbols (n)':<28}: {N_SYMBOLS:<26}║")
print(f"║  {'Merge steps':<28}: {len(merge_log):<26}║")
min_code = min(CODES, key=lambda x: len(CODES[x]))
max_code = max(CODES, key=lambda x: len(CODES[x]))
print(f"║  {'Shortest code':<28}: {CODES[min_code]} ({min_code}){'':<18}║")
print(f"║  {'Longest code':<28}: {CODES[max_code]} ({max_code}){'':<18}║")
print("╠" + "═"*W + "╣")
print("║  COMPRESSION" + " "*(W-13) + "║")
print(f"║  {'Message length':<28}: {len(MESSAGE)} chars{'':<20}║")
print(f"║  {'Huffman bits':<28}: {huff_bits:<26}║")
print(f"║  {'Fixed-length bits':<28}: {fixed_msg_bits} ({fixed_bits} × {len(MESSAGE)}){'':<10}║")
print(f"║  {'Compression ratio':<28}: {ratio:.3f}×{'':<22}║")
print(f"║  {'Round-trip correct':<28}: {'YES ✅' if match else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  INFORMATION THEORY" + " "*(W-20) + "║")
print(f"║  {'Shannon entropy':<28}: {entropy:.4f} bits/symbol{'':<12}║")
print(f"║  {'Huffman avg bits/sym':<28}: {avg_huff:.4f} bits/symbol{'':<12}║")
print(f"║  {'Prefix-free':<28}: {'YES ✅' if prefix_free else 'NO ❌':<26}║")
print(f"║  {'Kraft sum':<28}: {kraft:.6f}{'':<19}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about greedy algorithms in this context?

*Your answer here:*

---

**Q2 — Why does the greedy choice work here?**
For one of the algorithms in this project, explain the *exchange argument* — why swapping the greedy choice for any other choice cannot improve the solution.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
